## Cargar y Procesar Datos CSV Con Pandas

El formato **CSV (Comma-Separated Values)** es el estándar universal más utilizado para almacenar e intercambiar datos tabulares en ciencia de datos e ingeniería de software debido a su simplicidad, ligereza y compatibilidad universal.

### Parámetros más importantes de `pd.read_csv()`:
- `filepath_or_buffer`: Ruta del archivo local o URL remota.
- `sep` o `delimiter`: Separador de columnas (por defecto `,`, pero puede ser `;`, `\t`, `|`).
- `encoding`: Codificación de caracteres (ej. `'utf-8'`, `'latin-1'`).
- `parse_dates`: Lista de columnas que deben convertirse automáticamente a formato datetime.
- `index_col`: Columna que actuará como índice del DataFrame.
- `usecols`: Lista de columnas específicas que se desean leer (ahorra memoria).
- `na_values`: Lista de valores adicionales a interpretar como NaN/nulos.

In [ ]:
%pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar el archivo CSV exportado previamente
df = pd.read_csv('data/dataset_ventas.csv', parse_dates=['fecha'])
df.head(10)

## Exploración Inicial de la Estructura de Datos

In [ ]:
# Dimensiones, columnas y tipos de datos
print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas\n")
print("Tipos de datos detectados:")
print(df.dtypes)

In [ ]:
# Información técnica general de memoria y valores no nulos
df.info()

In [ ]:
# Estadísticas descriptivas de variables numéricas
df.describe()

In [ ]:
# Estadísticas de variables categóricas (texto)
df.describe(include='object')

## Limpieza, Validación e Ingeniería de Características

In [ ]:
# Verificar valores nulos y registros duplicados
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nRegistros duplicados encontrados: {df.duplicated().sum()}")

In [ ]:
# Extracción de características temporales a partir de la columna 'fecha'
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
df['nombre_mes'] = df['fecha'].dt.month_name()
df['dia_semana'] = df['fecha'].dt.day_name()

df[['fecha', 'año', 'mes', 'nombre_mes', 'dia_semana']].head()

## Filtrado y Consultas Analíticas

In [ ]:
# 1. Filtrar transacciones en 'Medellín' con ventas mayores a 1,500,000
ventas_medellin_altas = df[(df['ciudad'] == 'Medellín') & (df['total_venta'] > 1500000)]
print(f"Cantidad de ventas altas en Medellín: {len(ventas_medellin_altas)}")
ventas_medellin_altas.head()

In [ ]:
# 2. Filtrado con método .query() para categorías 'Tecnología' y 'Alimentos' con cantidad >= 10
ventas_tec_alim = df.query("categoria in ['Tecnología', 'Alimentos'] and cantidad >= 10")
print(f"Registros encontrados: {len(ventas_tec_alim)}")
ventas_tec_alim.head()

## Agrupaciones y Resúmenes Estadísticos (`groupby` & `pivot_table`)

In [ ]:
# Total de ventas, promedio y cantidad de operaciones por ciudad
resumen_ciudad = df.groupby('ciudad')['total_venta'].agg(
    total_ventas='sum',
    promedio_venta='mean',
    conteo_transacciones='count'
).reset_index().sort_values(by='total_ventas', ascending=False)

resumen_ciudad

In [ ]:
# Métricas de ventas agrupadas por categoría de producto
resumen_categoria = df.groupby('categoria').agg(
    unidades_vendidas=('cantidad', 'sum'),
    total_ingresos=('total_venta', 'sum'),
    precio_promedio=('precio_unitario', 'mean')
).reset_index().sort_values(by='total_ingresos', ascending=False)

resumen_categoria

In [ ]:
# Tabla Dinámica (Pivot Table): Ciudad vs Categoría de Producto
pivot_ventas = pd.pivot_table(
    df,
    values='total_venta',
    index='ciudad',
    columns='categoria',
    aggfunc='sum',
    fill_value=0
)
pivot_ventas

## Exportar Resultados y Reportes a CSV

In [ ]:
# Guardar los reportes generados a nuevos archivos CSV
resumen_ciudad.to_csv('data/reporte_ventas_por_ciudad.csv', index=False)
pivot_ventas.to_csv('data/reporte_pivot_ciudad_categoria.csv')

print("Reportes exportados exitosamente en la carpeta 'data/':")
print("  - data/reporte_ventas_por_ciudad.csv")
print("  - data/reporte_pivot_ciudad_categoria.csv")